## inertial calc

This notebook calculates the moment of inertia for different body segments using geometrical approximations and mass disstributions from the ANSUR dataset.

In [1]:
import pandas as pd

from anthropmass.mass.measurements_heights_module import *
from anthropmass.mass.volume_calculation_module import *
from anthropmass.mass.massclauser_module import *

Then we define the 'calculate_inertia' function.

This function computes the moment of ienrtia for different body parts using segment volumes, mass distribution and anthropometric measurements.

In [2]:

def calculate_inertia(Ansur, inputweight, inputheight):
    
    # volumes, mass, and measurements
    volumes = get_volumes(pd.DataFrame([Ansur]), inputheight)
    mass = clauser(Ansur, inputweight, inputheight)
    measurements = get_measurements(pd.DataFrame([Ansur]), inputheight).iloc[0]

    # Mass of each segment
    mH = mass["mH"]
    mLa = mass["mLA"]
    mUA = mass["mUA"]
    mF = mass["mF"]
    mS = mass["mS"]
    mTh = mass["mTh"]
    mHe = mass["mHe"]
    mTr = mass["mTr"]

    # Volume of each trunk segment
    vUT = volumes["vUT"]
    vMT = volumes["vMT"]
    vLT = volumes["vLT"]

    # Measurements of the segments
    a1, b1, h1 = measurements["a1"], measurements["b1"], measurements["h1"]
    a2, b2, h2 = measurements["a2"], measurements["b2"], measurements["h2"]
    a3, b3 = measurements["a3"], measurements["b3"]
    a4, b4, h3 = measurements["a4"], measurements["b4"], measurements["h3"]
    r1thigh, r2thigh, h4 = measurements["r1thigh"], measurements["r2thigh"], measurements["h4"]
    r1shank, r2shank, h6 = measurements["r1shank"], measurements["r2shank"], measurements["h6"]
    a5, b5, a6, b6, h7 = measurements["a5"], measurements["b5"], measurements["a6"], measurements["b6"], measurements["h7"]
    r1, r2, h8 = measurements["r1"], measurements["r2"], measurements["h8"]
    c7, h9, b7 = measurements["c7"], measurements["h9"], measurements["b7"]
    a7, b8 = measurements["a7"], measurements["b8"]
    r3, r4, h10 = measurements["r3"], measurements["r4"], measurements["h10"]

    # Calculation of each trunk mass
    Tot_trunk_vol = vUT + vMT + vLT
    mUpTr = (vUT/Tot_trunk_vol) * mTr
    mMdTr = (vMT/Tot_trunk_vol) * mTr
    mLwTr = (vLT/Tot_trunk_vol) * mTr

    # Storing inertia results
    inertia = {}

    # Upper trunk, elliptical cylinder
    inertia["Izz_UT"] = 0.5 * mUpTr * (a1**2 + b1**2)
    inertia["Ixx_UT"] = 0.25 * mUpTr * (a1**2 + (h1**2)/3)
    inertia["Iyy_UT"] = 0.25 * mUpTr * (b1**2 + (h1**2)/3)

    # Lower trunk, elliptical cylinder
    inertia["Izz_LT"] = 0.5 * mLwTr * (a4**2 + b4**2)
    inertia["Ixx_LT"] = 0.25 * mLwTr * (a4**2 + (h3**2)/3)
    inertia["Iyy_LT"] = 0.25 * mLwTr * (b4**2 + (h3**2)/3)

    # Thigh, frustum
    inertia["Izz_T"] = (mTh/10) * (r1thigh**2 + r1thigh * r2thigh + r2thigh**2)
    inertia["Ixx_T"] = inertia["Iyy_T"] = (mTh/20) * (3 * r1thigh**2 + 3 * r2thigh**2 + 4 * h4**2)

    # Shank, frustum
    inertia["Izz_S"] = (mS/10) * (r1shank**2 + r1shank * r2shank + r2shank**2)
    inertia["Ixx_S"] = inertia["Iyy_S"] = (mS/20) * (3 * r1shank**2 + 3 * r2shank**2 + 4 * h6**2)

    # Foot, frustum            
    #inertia["Ixx_F"] = (mF/10) * (a6**2 + a6 * a5 + a5**2)
    #inertia["Izz_F"] = inertia["Iyy_F"] = (mF/20) * (3 * a6**2 + 3 * a5**2 + 4 * h7**2)

    # Upper arm, frustum
    inertia["Izz_UA"] = (mUA/10) * (r1**2 + r1 * r2 + r2**2)
    inertia["Ixx_UA"] = inertia["Iyy_UA"] = (mUA/20) * (3* r1**2 + 3 * r2**2 + 4 * h8**2)

    # Lower arm, frustum
    inertia["Izz_LA"] = (mLa/10) * (r3**2 + r3 * r4 + r4**2)
    inertia["Ixx_LA"] = inertia["Iyy_LA"] = (mLa/20) * (3* r3**2 + 3 * r4**2 + 4 * h10**2)

    # Hand, ellipsoid
    inertia["Ixx_Ha"] = (mH/5) * (b7**2 + c7**2)
    inertia["Iyy_Ha"] = (mH/5) * (h9**2 + c7**2)
    inertia["Izz_Ha"] = (mH/5) * (h9**2 + b7**2)

    # Head, ellipsoid
    inertia["Ixx_H"] = (mHe/5) * (b8**2 + a7**2)
    inertia["Iyy_H"] = (mHe/5) * (b8**2 + a7**2)
    inertia["Izz_H"] = (mHe/5) * (b8**2 + b8**2)

    #The following section calculates the inertia of complex shapes using frustum of an elliptical cone geometry for the middle trunk and foot.

    # Middle trunk, frustum
    B1_ab = (a2 - a3) * (b2 - b3)
    B2_ab = a3 * (b2 - b3) + b3 * (a2 - a3)
    B3_ab = a3 * b3

    B1_bbb = (b2 - b3) ** 3
    B2_bbb = 3 * b3 * (b2 - b3) ** 2
    B3_bbb = 3 * b3**2 * (b2 - b3)

    B1_aaa = (a2 - a3) ** 3
    B2_aaa = 3 * a3 * (a2 - a3)**2
    B3_aaa = 3 * a3**2 * (a2 - a3)

    B1_aabb = (a2 - a3)**2 * (b2 - b3)**2
    B2_aabb = a3 * (a2 - a3) * (b2 - b3)**2 + b3 * (b2 - b3) * (a2 - a3)**2
    B3_aabb = a3 * b3 * (a2 - a3) * (b2 - b3)

    A1 = (1/3) * B1_ab + (1/2) * B2_ab + B3_ab
    A2 = (1/4) * B1_ab + (1/3) * B2_ab + (1/2) * B3_ab
    A3_bbb = (1/5) * B1_bbb + (1/4) * B2_bbb + (1/3) * B3_bbb
    A3_aaa = (1/5) * B1_aaa + (1/4) * B2_aaa + (1/3) * B3_aaa
    A3_aabb = (1/5) * B1_aabb + (1/4) * B2_aabb + (1/3) * B3_aabb

    inertia["Ixx_MT"] = (1/4) * mMdTr * (A3_bbb / A1) + mMdTr * h2**2 * (A2 / A1) - mMdTr * (h2 * A2 / A1)**2
    inertia["Iyy_MT"] = (1/4) * mMdTr * (A3_aaa / A1) + mMdTr * h2**2 * (A2 / A1) - mMdTr * (h2 * A2 / A1)**2
    inertia["Izz_MT"] = (1/4) * mMdTr * (A3_aabb / A1)

    #Foot, frustum of an elliptical solid
    B1_ab_F = (a5 - a6) * (b5 - b6)
    B2_ab_F = a6 * (b5 - b6) + b6 * (a5 - a6)
    B3_ab_F = a6 * b6

    B1_bbb_F = (b5 - b6) ** 3
    B2_bbb_F = 3 * b6 * (b5 - b6) ** 2
    B3_bbb_F = 3 * b6**2 * (b5 - b6)

    B1_aaa_F = (a5 - a6) ** 3
    B2_aaa_F = 3 * a6 * (a5 - a6)**2
    B3_aaa_F = 3 * a6**2 * (a5 - a6)

    B1_aabb_F = (a5 - a6)**2 * (b5 - b6)**2
    B2_aabb_F = a6 * (a5 - a6) * (b5 - b6)**2 + b6 * (b5 - b6) * (a5 - a6)**2
    B3_aabb_F = a6 * b6 * (a5 - a6) * (b5 - b6)

    #A terms
    A1_F = (1/3) * B1_ab_F + (1/2) * B2_ab_F + B3_ab_F
    A2_F = (1/4) * B1_ab_F + (1/3) * B2_ab_F + (1/2) * B3_ab_F
    A3_bbb_F = (1/5) * B1_bbb_F + (1/4) * B2_bbb_F + (1/3) *B3_bbb_F
    A3_aaa_F = (1/5) * B1_aaa_F + (1/4) * B2_aaa_F + (1/3) * B3_aaa_F
    A3_aabb_F = (1/5) * B1_aabb_F + (1/4) * B2_aabb_F + (1/3) *B3_aabb_F

    inertia["Ixx_F"] = (1/4) * mF *(A3_bbb_F/A1_F)
    inertia["Iyy_F"] = (1/4) * mF *(A3_aaa_F/A1_F) + mF * h7**2 *(A2_F/A1_F) - mF *(h7 * A2_F/A1_F)**2
    inertia["Izz_F"] = (1/4) * mF *(A3_aabb_F/A1_F) + mF * h7**2 *(A2_F/A1_F) - mF *(h7 * A2_F/A1_F)**2

    return inertia

Example: Calculation of inertia for one individual

In [3]:
#FOR INDIVIDUAL 

#Ansur = pd.read_csv("/Users/parsakhodabakhsh/PYTHON wow/KANDIDAT/bsp_estimation/ANSURIIFEMALEPublic.csv", encoding='latin1')

#Ansur_row = Ansur.iloc[0]  # Get Series

#inputweight = Ansur_row["weightkg"]
#inputheight = Ansur_row["stature"]

# Wrap Ansur_row as DataFrame only for those two functions
#volumes = get_volumes(pd.DataFrame([Ansur_row]), inputheight)
#mass = clauser(Ansur_row, inputweight, inputheight)
#measurements = get_measurements(pd.DataFrame([Ansur_row]), inputheight).iloc[0]

# Then call the rest of your function normally...
#inertia = calculate_inertia(Ansur_row, inputweight, inputheight)

# Print results
#for part, I_value in inertia.items():
    #print(f"{part}: {I_value}")

Process full dataset and compute average inertias.
This section looås through the entire ANSUR dataset, calculates inertia for each individual and then computes the avrage for each segment.

In [ ]:
#AVRAGE

# Load the full dataset
ansur_df = pd.read_csv("/Users/parsakhodabakhsh/PYTHON wow/KANDIDAT/bsp_estimation/ANSURIIMALEPublic.csv", encoding='latin1')

# Store all inertia results
inertia_records = []

# Loop through each person
for _, row in ansur_df.iterrows():
    inputweight = row["weightkg"]
    inputheight = row["stature"]
    
    try:
        inertia = calculate_inertia(row, inputweight, inputheight)
        inertia_records.append(inertia)
    except Exception as e:
        print(f"Skipping due to error: {e}")

# Convert list of dicts to DataFrame
inertia_df = pd.DataFrame(inertia_records)

# Calculate average moment of inertia for each body part
avg_inertia = inertia_df.mean()

# Print results
print("\nAverage Moment of Inertia (kg·m²):")
for part, val in avg_inertia.items():
    print(f"{part}: {val:.4f}")

FileNotFoundError: [Errno 2] No such file or directory: '/Users/parsakhodabakhsh/PYTHON wow/KANDIDAT/bsp_estimation/ANSURIIMALEPublic.csv'